# EDA: property listings & TS RERA projects

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW = "../data/raw"
PROCESSED = "../data/processed"
pd.set_option("display.max_columns", 50)

## Listings (nobroker.in / squareyards.com)

In [ ]:
listings = pd.read_csv(f"{RAW}/All_Merged_Updated.csv", low_memory=False)
listings.columns = [c.strip().lstrip("\ufeff") for c in listings.columns]
listings.shape

In [ ]:
missing = (listings.isna().mean() * 100).round(1).sort_values(ascending=False)
missing.head(12).plot.barh(figsize=(6,4), color="#2a78d6")
plt.title("Missing % by column (top 12)")
plt.gca().invert_yaxis()
plt.tight_layout()

In [ ]:
def site(u):
    if not isinstance(u, str):
        return "unknown"
    if "nobroker" in u:
        return "nobroker"
    if "squareyards" in u:
        return "squareyards"
    if "magicbricks" in u:
        return "magicbricks"
    return "other"

listings["_site"] = listings["URL"].map(site)
listings["_site"].value_counts()

In [ ]:
listings["BHK"].value_counts().sort_index().plot.bar(figsize=(6,4), color="#2a78d6")
plt.title("BHK distribution")
plt.tight_layout()

In [ ]:
top_loc = (listings.groupby("Locality")["PricePerSqft"]
           .agg(["count", "median"])
           .query("count >= 20")
           .sort_values("median", ascending=False)
           .head(15))
top_loc["median"].plot.barh(figsize=(6,5), color="#2a78d6")
plt.title("Top localities by median price/sqft")
plt.gca().invert_yaxis()
plt.tight_layout()

In [ ]:
dup_urls = listings["URL"].duplicated().sum()
print(f"duplicate rows: {listings.duplicated().sum():,}")
print(f"duplicate URLs: {dup_urls:,} ({dup_urls/len(listings):.1%})")
listings["PricePerSqft"].describe(percentiles=[.01,.25,.5,.75,.95,.99])

## TS RERA registered projects

In [ ]:
rera = pd.read_csv(f"{RAW}/projects_from_jsonl.csv", low_memory=False)
rera.shape

In [ ]:
rera["district"].value_counts().head(10).plot.barh(figsize=(6,4), color="#2a78d6")
plt.title("RERA projects by district (top 10)")
plt.gca().invert_yaxis()
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
rera["Project Information :: Project Status"].value_counts().plot.bar(ax=axes[0], color="#2a78d6")
axes[0].set_title("Project status")
rera["Project Information :: Project Type"].value_counts().plot.bar(ax=axes[1], color="#2a78d6")
axes[1].set_title("Project type")
plt.tight_layout()

In [ ]:
rera["summary.Promoter Name"].value_counts().head(10).plot.barh(figsize=(6,4), color="#2a78d6")
plt.title("Top promoters by project count")
plt.gca().invert_yaxis()
plt.tight_layout()
print("unique promoters:", rera["summary.Promoter Name"].nunique())

**Data quality notes**\n\n- `Society` in listings uses common neighbourhood names; RERA `Address Details :: Locality` uses formal village/mandal names — not directly joinable.\n- `Pincode` overlaps well between the two sources and is used as the merge blocking key (see `02_merge.ipynb`).\n- Listings' `Transaction` column is not a clean Sale/Rent label — deal type has to be derived from the URL slug instead (see `03_modeling_dataset.ipynb`).